In [ ]:
import os, re, random, json
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from difflib import SequenceMatcher

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
INPUT_CSV = "biasmedqa_bias_pairs.csv"
OUTPUT_DIR = None 
MAX_NEW_TOKENS = 200
DO_SAMPLE = False
TEMPERATURE = 0.0


TOP_K_LIST = [5, 10, 15, 20, 25]
ALPHA_LIST = [0.5, 1.0]

DIRECTION_ALPHA = 0.5
USE_4BIT = True

SMOKE_TEST = False  #for sample test


In [ ]:
print("=" * 60)
print("MODEL_ID =", MODEL_ID)
print("INPUT_CSV =", INPUT_CSV)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("=" * 60)

In [ ]:
# config
if "MODEL_NAME" in globals() and MODEL_NAME is not None:
    MODEL_ID = MODEL_NAME

MODEL_TAG = MODEL_ID.split("/")[-1].replace("-", "_").replace(".", "_")
K_TAG = "k" + "_".join(str(k) for k in TOP_K_LIST)

if OUTPUT_DIR is None or str(OUTPUT_DIR).strip() == "":
    OUTPUT_DIR = f"bmqa_full_run_{MODEL_TAG}_{K_TAG}"

os.makedirs(OUTPUT_DIR, exist_ok=True)

RUN_MANIFEST_PATH = os.path.join(OUTPUT_DIR, "run_manifest.json")
_run_manifest = {
    "model_id": MODEL_ID,
    "input_csv": os.path.abspath(INPUT_CSV),
    "top_k_list": [int(k) for k in TOP_K_LIST],
}
if os.path.exists(RUN_MANIFEST_PATH):
    with open(RUN_MANIFEST_PATH) as _f:
        _old_manifest = json.load(_f)
    if _old_manifest != _run_manifest:
        raise RuntimeError(
            "OUTPUT_DIR contains checkpoints from a different run. "
            f"existing={_old_manifest}, current={_run_manifest}. "
            "Use a new OUTPUT_DIR; do not reuse these CSVs."
        )
else:
    with open(RUN_MANIFEST_PATH, "w") as _f:
        json.dump(_run_manifest, _f, indent=2)

print("MODEL_ID:", MODEL_ID)
print("MODEL_TAG:", MODEL_TAG)
print("INPUT_CSV:", INPUT_CSV)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("TOP_K_LIST:", TOP_K_LIST)


In [ ]:

#Load model/tokenizer
def load_model_and_tokenizer(model_id: str, use_4bit: bool = True):
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    kwargs = {"device_map": {"": 0}, "trust_remote_code": True}
    kwargs["torch_dtype"] = torch.float16 if torch.cuda.is_available() else torch.float32

    if use_4bit and torch.cuda.is_available():
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )

    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.eval()
    return model, tokenizer

model, tokenizer = load_model_and_tokenizer(MODEL_ID, USE_4BIT)
print("Model loaded on:", next(model.parameters()).device)

In [ ]:
# Model adapter
def get_transformer_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    raise ValueError("Could not find transformer layers.")

def get_attn_output_module(layer):
    if hasattr(layer, "self_attn") and hasattr(layer.self_attn, "o_proj"):
        return layer.self_attn.o_proj
    if hasattr(layer, "attn") and hasattr(layer.attn, "c_proj"):
        return layer.attn.c_proj
    raise ValueError("Could not find attention output projection module.")

layers = get_transformer_layers(model)
num_layers = len(layers)
num_heads = int(model.config.num_attention_heads)
_o_proj0 = get_attn_output_module(layers[0])
attention_width = int(getattr(_o_proj0, "in_features", 0))
configured_head_dim = getattr(model.config, "head_dim", None)
if configured_head_dim is not None:
    head_dim = int(configured_head_dim)
elif attention_width and attention_width % num_heads == 0:
    head_dim = attention_width // num_heads
else:
    head_dim = int(model.config.hidden_size) // num_heads

if attention_width and head_dim * num_heads != attention_width:
    raise RuntimeError(
        f"Bad head geometry for {MODEL_ID}: heads={num_heads}, "
        f"head_dim={head_dim}, heads*head_dim={num_heads*head_dim}, "
        f"o_proj.in_features={attention_width}."
    )
print("layers", num_layers, "heads", num_heads, "head_dim", head_dim,
      "attention_width", attention_width)


In [ ]:
#Load prompt-pair CSV
REQUIRED_COLUMNS = ["seed_id", "pair_type", "comparison", "d0_demographic", "d1_demographic", "prompt_d0", "prompt_d1"]

def normalize_demographic_text(x: str) -> str:
    return str(x).replace("_", " ").strip()

def load_pair_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}; found {list(df.columns)}")
    df = df.copy()
    df["pair_id"] = (df["seed_id"].astype(str) + "_" + df["pair_type"].astype(str) + "_" +
                     df["comparison"].astype(str) + "_" + df["d0_demographic"].astype(str) +
                     "_to_" + df["d1_demographic"].astype(str))
    df["attribute"] = df["pair_type"].astype(str).str.lower().str.strip()
    df["group0"] = df["d0_demographic"].apply(normalize_demographic_text)
    df["group1"] = df["d1_demographic"].apply(normalize_demographic_text)
    df["prompt_d0"] = df["prompt_d0"].fillna("").astype(str)
    df["prompt_d1"] = df["prompt_d1"].fillna("").astype(str)
    if ((df["prompt_d0"].str.len() == 0) | (df["prompt_d1"].str.len() == 0)).any():
        raise ValueError("Some rows have empty prompts.")
    return df.reset_index(drop=True)

pairs_df = load_pair_csv(INPUT_CSV)
print("Rows:", len(pairs_df), "Unique seeds:", pairs_df["seed_id"].nunique())
print(pairs_df.groupby(["attribute", "comparison"]).size())
display(pairs_df.head(3))

In [ ]:
TARGET_N_PAIRS = globals().get("TARGET_N_PAIRS", 100) 
SELECT_SEED = 42          
SUBSET_TAG = f"n{TARGET_N_PAIRS}"   
 
 
def _dominant_attribute_per_seed(df):
    counts = (df.groupby(["seed_id", "attribute"])["pair_id"]
                .nunique().reset_index())
    dom = (counts.sort_values(["seed_id", "pair_id"], ascending=[True, False])
                 .drop_duplicates("seed_id")
                 .set_index("seed_id")["attribute"].to_dict())
    return dom
 
 
def build_nested_seed_order(df, seed=SELECT_SEED):
    rng = np.random.default_rng(seed)
    dom = _dominant_attribute_per_seed(df)
 
    buckets = {}
    for s, a in dom.items():
        buckets.setdefault(a, []).append(s)
    for a in buckets:
        arr = np.array(sorted(buckets[a]))          # sort first for determinism
        rng.shuffle(arr)                            # then shuffle with fixed rng
        buckets[a] = list(arr)

    order = []
    attrs = sorted(buckets.keys())
    i = 0
    while any(buckets[a] for a in attrs):
        a = attrs[i % len(attrs)]
        if buckets[a]:
            order.append(buckets[a].pop(0))
        i += 1
    return order
 
 
def select_pairs_by_seed(df, target_n_pairs=TARGET_N_PAIRS, seed=SELECT_SEED, verbose=True):
    order = build_nested_seed_order(df, seed=seed)
    pairs_per_seed = df.groupby("seed_id")["pair_id"].nunique().to_dict()
 
    chosen, running = [], 0
    for s in order:
        if running >= target_n_pairs:
            break
        chosen.append(s)
        running += pairs_per_seed.get(s, 0)
 
    sel = df[df["seed_id"].isin(chosen)].reset_index(drop=True)
 
    if verbose:
        print(f"Requested ~{target_n_pairs} pairs -> selected {sel['pair_id'].nunique()} "
              f"pairs from {len(chosen)} seeds (of {df['seed_id'].nunique()} available).")
        print("Per attribute:")
        print(sel.groupby("attribute")["pair_id"].nunique().to_string())
        print("Per attribute x comparison:")
        print(sel.groupby(["attribute", "comparison"])["pair_id"].nunique().to_string())
    return sel, chosen, order
 
 
pairs_df_full = pairs_df.copy()                     #full set
pairs_df, selected_seeds, seed_order = select_pairs_by_seed(pairs_df_full, TARGET_N_PAIRS, SELECT_SEED)

_sel_path = os.path.join(OUTPUT_DIR, f"selected_seeds_{SUBSET_TAG}.json")
with open(_sel_path, "w") as f:
    json.dump({
        "target_n_pairs": TARGET_N_PAIRS,
        "select_seed": SELECT_SEED,
        "n_selected_pairs": int(pairs_df["pair_id"].nunique()),
        "selected_seeds": [str(s) for s in selected_seeds],
        "full_seed_order": [str(s) for s in seed_order],  
    }, f, indent=2)
print("Saved selection ->", _sel_path)

In [ ]:

#Generation + lightweight TE_sem
@torch.no_grad()
def generate_text(prompt: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=DO_SAMPLE,
        pad_token_id=tokenizer.eos_token_id,
    )
    if DO_SAMPLE:
        gen_kwargs["temperature"] = TEMPERATURE
    out = model.generate(**inputs, **gen_kwargs)
    new_ids = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


def normalize_for_metric(text: str) -> str:
    """Light normalization for response-level divergence."""
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def token_set(text: str):
    """Simple token set after removing very short tokens."""
    text = normalize_for_metric(text)
    toks = re.findall(r"[a-zA-Z0-9]+", text)
    return set(t for t in toks if len(t) > 1)


def jaccard_divergence(a: str, b: str) -> float:
    """1 - Jaccard similarity over word tokens."""
    A, B = token_set(a), token_set(b)
    if len(A | B) == 0:
        return 0.0
    return float(1.0 - len(A & B) / len(A | B))


def sequence_divergence(a: str, b: str) -> float:
    """1 - character/sequence similarity."""
    a = normalize_for_metric(a)
    b = normalize_for_metric(b)
    return float(1.0 - SequenceMatcher(None, a, b).ratio())


def compute_te_sem(y0: str, y1: str) -> float:
    return 0.5 * jaccard_divergence(y0, y1) + 0.5 * sequence_divergence(y0, y1)


def compute_quality_drift(y0_orig: str, y1_orig: str, y0_after: str, y1_after: str):
    drift0 = compute_te_sem(y0_orig, y0_after)
    drift1 = compute_te_sem(y1_orig, y1_after)
    return 0.5 * (drift0 + drift1), drift0, drift1



def compute_bias_gap(y0: str, y1: str):
    s0, s1 = extract_likert_score(y0), extract_likert_score(y1)
    if np.isnan(s0) or np.isnan(s1):
        return np.nan, s0, s1
    return abs(s0 - s1), s0, s1


In [ ]:
#test run before any large scan
_probe = pairs_df.iloc[0]
_probe_inputs = tokenizer(_probe["prompt_d0"], return_tensors="pt").to(model.device)
with torch.no_grad():
    _probe_out = model(**_probe_inputs)
assert torch.isfinite(_probe_out.logits).all(), "Non-finite baseline logits"
_probe_text = generate_text(_probe["prompt_d0"], max_new_tokens=8)
print("Preflight model:", MODEL_ID)
print("Preflight continuation:", repr(_probe_text))
if not _probe_text:
    print("WARNING: model emitted EOS immediately on the first prompt. ")



In [ ]:

# 6. Baseline generation
baseline_path = os.path.join(OUTPUT_DIR, "baseline_outputs.csv")

def _baseline_covers_selection(path, pairs_df):
    if not os.path.exists(path):
        return False
    try:
        prev = pd.read_csv(path)
    except Exception:
        return False
    if "pair_id" not in prev.columns:
        return False
    need = set(pairs_df["pair_id"])
    have = set(prev["pair_id"].astype(str))
    missing = need - have
    if missing:
        print(f"[baseline] stale/incomplete checkpoint: {len(missing)} of "
              f"{len(need)} pairs missing -> regenerating. e.g. {list(missing)[:2]}")
        return False
    return True

if _baseline_covers_selection(baseline_path, pairs_df):
    baseline_df = pd.read_csv(baseline_path)
    baseline_df["pair_id"] = baseline_df["pair_id"].astype(str)
    # keep only the currently-selected pairs, in case the CSV has extras
    baseline_df = baseline_df[baseline_df["pair_id"].isin(set(pairs_df["pair_id"]))].reset_index(drop=True)
    print(f"[baseline] reusing checkpoint: {len(baseline_df)} pairs")
else:
    rows = []
    for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc="Baseline"):
        y0, y1 = generate_text(row["prompt_d0"]), generate_text(row["prompt_d1"])
        te_sem = compute_te_sem(y0, y1)
        bg, s0, s1 = compute_bias_gap(y0, y1)
        rows.append({**row.to_dict(), "y_d0": y0, "y_d1": y1, "TE_sem_original": te_sem,
                     "BiasGap_original": bg, "score_d0_original": s0, "score_d1_original": s1})
        pd.DataFrame(rows).to_csv(baseline_path, index=False)
    baseline_df = pd.DataFrame(rows)

display(baseline_df[["pair_id", "attribute", "TE_sem_original", "BiasGap_original", "score_d0_original", "score_d1_original"]].head())
print("BiasGap parse rate:", baseline_df["BiasGap_original"].notna().mean())

In [ ]:
#Activation cache + NIE utilities
def js_divergence_from_logits(logits_p: torch.Tensor, logits_q: torch.Tensor) -> float:
    p = F.softmax(logits_p.float(), dim=-1)
    q = F.softmax(logits_q.float(), dim=-1)
    m = 0.5 * (p + q)
    js = 0.5 * (F.kl_div(m.log(), p, reduction="sum") + F.kl_div(m.log(), q, reduction="sum"))
    return float(js.item())

@torch.no_grad()
def get_logits(prompt: str):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model(**inputs)
    return out.logits[:, -1, :].detach()

def cache_o_proj_input(prompt: str, layer_idx: int):
    module = get_attn_output_module(layers[layer_idx])
    cache = {}
    def pre_hook(mod, inputs):
        cache["x"] = inputs[0].detach()
    handle = module.register_forward_pre_hook(pre_hook)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        _ = model(**inputs)
    handle.remove()
    return cache["x"]

def run_single_head_patch_logits(prompt_target: str, layer_idx: int, head_idx: int, source_activation: torch.Tensor):
    module = get_attn_output_module(layers[layer_idx])
    h0, h1 = head_idx * head_dim, (head_idx + 1) * head_dim
    def pre_hook(mod, inputs):
        x = inputs[0]
        x_new = x.clone()
        x_new[:, -1:, h0:h1] = source_activation[:, -1:, h0:h1].to(
            x_new.device, dtype=x_new.dtype)
        return (x_new,) + inputs[1:]
    handle = module.register_forward_pre_hook(pre_hook)
    inputs = tokenizer(prompt_target, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model(**inputs)
    handle.remove()
    return out.logits[:, -1, :].detach()

In [ ]:
# NIE scan
nie_path = os.path.join(OUTPUT_DIR, "head_level_NIE.csv")

def _nie_covers_selection(path, pairs_df):
    if not os.path.exists(path):
        return False
    try:
        prev = pd.read_csv(path, usecols=["pair_id", "layer", "head"])
    except Exception:
        return False
    need = set(pairs_df["pair_id"].astype(str))
    prev["pair_id"] = prev["pair_id"].astype(str)
    have = set(prev["pair_id"])
    missing = need - have
    expected_per_pair = num_layers * num_heads
    counts = prev[prev["pair_id"].isin(need)].groupby("pair_id").size()
    partial = [pid for pid in need if counts.get(pid, 0) != expected_per_pair]
    if missing or partial:
        print(f"[NIE] stale/incomplete checkpoint: missing={len(missing)}, "
              f"partial={len(partial)} -> rescanning.")
        return False
    return True

if _nie_covers_selection(nie_path, pairs_df):
    nie_df = pd.read_csv(nie_path)
    nie_df["pair_id"] = nie_df["pair_id"].astype(str)
    nie_df = nie_df[nie_df["pair_id"].isin(set(pairs_df["pair_id"]))].reset_index(drop=True)
    print(f"[NIE] reusing checkpoint: {nie_df['pair_id'].nunique()} pairs")
else:
    nie_rows = []
    for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc="NIE scan"):
        logits_d0 = get_logits(row["prompt_d0"])
        source_cache_by_layer = {l: cache_o_proj_input(row["prompt_d1"], l) for l in range(num_layers)}
        for l in range(num_layers):
            src = source_cache_by_layer[l]
            for h in range(num_heads):
                patched_logits = run_single_head_patch_logits(row["prompt_d0"], l, h, src)
                nie = js_divergence_from_logits(logits_d0, patched_logits)
                nie_rows.append({"pair_id": row["pair_id"], "seed_id": row["seed_id"], "attribute": row["attribute"],
                                 "comparison": row["comparison"], "layer": l, "head": h, "NIE": nie})
        pd.DataFrame(nie_rows).to_csv(nie_path, index=False)
    nie_df = pd.DataFrame(nie_rows)

display(nie_df.head())

In [ ]:
#Select top-K + shared circuit
def select_topk_heads(nie_df, attribute=None, k=10):
    k = int(k) 
    df = nie_df.copy()
    if attribute is not None:
        df = df[df["attribute"].str.lower() == attribute.lower()]
    if len(df) == 0:
        empty = pd.DataFrame(columns=["layer", "head", "NIE"])
        return [], empty

    agg = (
        df.groupby(["layer", "head"], as_index=False)["NIE"]
        .mean()
        .sort_values("NIE", ascending=False)
        .head(k)
        .reset_index(drop=True)
    )
    heads = [(int(r["layer"]), int(r["head"])) for _, r in agg.iterrows()]
    return heads, agg


def jaccard(a, b):
    A, B = set(a), set(b)
    return len(A & B) / len(A | B) if len(A | B) else np.nan


all_top_heads = {}
TOP_K_LIST = [int(k) for k in TOP_K_LIST]  

for K in TOP_K_LIST:
    top_all_k, top_all_df_k = select_topk_heads(nie_df, None, K)
    top_race_k, top_race_df_k = select_topk_heads(nie_df, "race", K)
    top_sex_k, top_sex_df_k = select_topk_heads(nie_df, "sex", K)

    shared_heads_k = sorted(list(set(top_race_k) & set(top_sex_k)))
    combined_union_heads_k = sorted(list(set(top_race_k) | set(top_sex_k)))

    all_top_heads[K] = {
        "all": top_all_k,
        "race": top_race_k,
        "sex": top_sex_k,
        "shared": shared_heads_k,
        "combined_union": combined_union_heads_k,
        "top_all_df": top_all_df_k,
        "top_race_df": top_race_df_k,
        "top_sex_df": top_sex_df_k,
        "jaccard_race_sex": jaccard(top_race_k, top_sex_k),
    }

    # Save per-K head lists for reproducibility.
    top_all_df_k.to_csv(os.path.join(OUTPUT_DIR, f"top_all_heads_K{K}.csv"), index=False)
    top_race_df_k.to_csv(os.path.join(OUTPUT_DIR, f"top_race_heads_K{K}.csv"), index=False)
    top_sex_df_k.to_csv(os.path.join(OUTPUT_DIR, f"top_sex_heads_K{K}.csv"), index=False)

    print(f"\nK={K}")
    print("  race:", top_race_k)
    print("  sex:", top_sex_k)
    print("  shared:", shared_heads_k)
    print("  Jaccard:", all_top_heads[K]["jaccard_race_sex"])

#first K only for display / quick heatmap labels.
DISPLAY_K = TOP_K_LIST[0]
top_all = all_top_heads[DISPLAY_K]["all"]
top_race = all_top_heads[DISPLAY_K]["race"]
top_sex = all_top_heads[DISPLAY_K]["sex"]
shared_heads = all_top_heads[DISPLAY_K]["shared"]
combined_union_heads = all_top_heads[DISPLAY_K]["combined_union"]
top_all_df = all_top_heads[DISPLAY_K]["top_all_df"]
top_race_df = all_top_heads[DISPLAY_K]["top_race_df"]
top_sex_df = all_top_heads[DISPLAY_K]["top_sex_df"]

display(top_race_df)
display(top_sex_df)


In [ ]:
#Multi-head interventions
def build_layer_caches(prompt: str, mediator_heads: List[Tuple[int, int]]):
    return {l: cache_o_proj_input(prompt, l) for l in sorted(set(l for l, h in mediator_heads))}

def build_neutral_caches(cache0, cache1):
    neutral = {}
    for l in cache0:
        a0, a1 = cache0[l], cache1[l]
        T = min(a0.shape[1], a1.shape[1])
        n = a0.clone()
        n[:, -T:, :] = 0.5 * (a0[:, -T:, :] + a1[:, -T:, :])
        neutral[l] = n.detach()
    return neutral

def build_directional_caches(cache0, cache1):
    direction = {}
    for l in cache0:
        a0, a1 = cache0[l], cache1[l]
        T = min(a0.shape[1], a1.shape[1])
        d = torch.zeros_like(a0)
        d[:, -T:, :] = a1[:, -T:, :] - a0[:, -T:, :]
        direction[l] = d.detach()
    return direction

@torch.no_grad()
def generate_with_multi_head_intervention(prompt, mediator_heads, mode, neutral_caches=None, direction_caches=None,
                                          alpha=0.5, direction_sign=1.0, max_new_tokens=MAX_NEW_TOKENS):
    assert mode in ["zero_ablation", "midpoint_intervention", "directional_intervention"]
    handles = []
    def make_hook(layer_idx):
        def pre_hook(mod, inputs):
            x = inputs[0]
            x_new = x.clone()
            for l, h in mediator_heads:
                if l != layer_idx: continue
                h0, h1 = h * head_dim, (h + 1) * head_dim
                if mode == "zero_ablation":
                    x_new[:, :, h0:h1] = 0
                elif mode == "midpoint_intervention":
                    neu = neutral_caches[layer_idx]
                    # Cached prompt positions are invalid during one-token KV decoding.
                    if x_new.shape[1] > 1:
                        T = min(x_new.shape[1], neu.shape[1])
                        x_new[:, -T:, h0:h1] = neu[:, -T:, h0:h1].to(
                            x_new.device, dtype=x_new.dtype)
                elif mode == "directional_intervention":
                    direction = direction_caches[layer_idx]
                    # Do not reuse position-0 prompt direction for every decode token.
                    if x_new.shape[1] > 1:
                        T = min(x_new.shape[1], direction.shape[1])
                        x_new[:, -T:, h0:h1] = (
                            x_new[:, -T:, h0:h1]
                            + direction_sign * alpha * direction[:, -T:, h0:h1].to(
                                x_new.device, dtype=x_new.dtype)
                        )
            return (x_new,) + inputs[1:]
        return pre_hook
    for l in sorted(set(l for l, h in mediator_heads)):
        handles.append(get_attn_output_module(layers[l]).register_forward_pre_hook(make_hook(l)))
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    gen_kwargs = dict(max_new_tokens=max_new_tokens, do_sample=DO_SAMPLE, pad_token_id=tokenizer.eos_token_id)
    if DO_SAMPLE: gen_kwargs["temperature"] = TEMPERATURE
    out = model.generate(**inputs, **gen_kwargs)
    for handle in handles: handle.remove()
    new_ids = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

In [ ]:
#Main mitigation: K sweep over attribute-specific circuits
def choose_circuit_for_k(attribute: str, K: int):
    info = all_top_heads[K]
    attr = str(attribute).lower().strip()
    if attr == "race":
        return info["race"]
    if attr == "sex":
        return info["sex"]
    return info["all"]


mitigation_path = os.path.join(OUTPUT_DIR, f"mitigation_results_{K_TAG}.csv")

def _mitigation_checkpoint_complete(path):
    if not os.path.exists(path):
        return False
    try:
        d = pd.read_csv(path)
    except Exception:
        return False
    required = {"K", "pair_id", "method", "alpha"}
    if not required.issubset(d.columns):
        return False
    #expected = len(TOP_K_LIST) * pairs_df["pair_id"].nunique() * 3
    expected = (len(TOP_K_LIST) * pairs_df["pair_id"].nunique() * (2 + len(ALPHA_LIST)))
    key_cols = ["K", "pair_id", "method", "alpha"]
    return len(d[key_cols].drop_duplicates()) == expected

if _mitigation_checkpoint_complete(mitigation_path):
    mitigation_df = pd.read_csv(mitigation_path)
else:
    rows = []
    base_lookup = baseline_df.set_index("pair_id")

    for K in TOP_K_LIST:
        print(f"\nRunning mitigation for K={K}")

        for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc=f"Mitigation K={K}"):
            pair_id = row["pair_id"]
            heads = choose_circuit_for_k(row["attribute"], K)

            if len(heads) == 0:
                heads = all_top_heads[K]["all"]

            cache0 = build_layer_caches(row["prompt_d0"], heads)
            cache1 = build_layer_caches(row["prompt_d1"], heads)
            neutral = build_neutral_caches(cache0, cache1)
            direction = build_directional_caches(cache0, cache1)

            base = base_lookup.loc[pair_id]
            y0_orig, y1_orig = str(base["y_d0"]), str(base["y_d1"])
            te_orig = float(base["TE_sem_original"])
            bg_orig = float(base["BiasGap_original"]) if not pd.isna(base["BiasGap_original"]) else np.nan

            outputs = {
                ("zero_ablation", np.nan): (
                    generate_with_multi_head_intervention(row["prompt_d0"], heads, "zero_ablation"),
                    generate_with_multi_head_intervention(row["prompt_d1"], heads, "zero_ablation"),
                ),
                ("midpoint_intervention", np.nan): (
                    generate_with_multi_head_intervention(row["prompt_d0"], heads, "midpoint_intervention", neutral_caches=neutral),
                    generate_with_multi_head_intervention(row["prompt_d1"], heads, "midpoint_intervention", neutral_caches=neutral),
                ),
            }
            for alpha in ALPHA_LIST:
                outputs[("directional_intervention",alpha)]=(
                    generate_with_multi_head_intervention(
                        row["prompt_d0"],
                        heads,
                        "directional_intervention",
                        direction_caches=direction,
                        alpha=alpha,
                        direction_sign=+1.0,
                    ),
                    generate_with_multi_head_intervention(
                        row["prompt_d1"],
                        heads,
                        "directional_intervention",
                        direction_caches=direction,
                        alpha=alpha,
                        direction_sign=-1.0,
                    ),
                )
                   

            for (method, alpha), (yd0, yd1) in outputs.items():
                te_after = compute_te_sem(yd0, yd1)
                bg_after, s0_after, s1_after = compute_bias_gap(yd0, yd1)
                drift_mean, drift0, drift1 = compute_quality_drift(y0_orig, y1_orig, yd0, yd1)

                rows.append({
                    "K": K,
                    "pair_id": pair_id,
                    "seed_id": row["seed_id"],
                    "attribute": row["attribute"],
                    "comparison": row["comparison"],
                    "group0": row["group0"],
                    "group1": row["group1"],
                    "method": method,
                    "num_mediator_heads": len(heads),
                    "mediator_heads": str(heads),
                    "method": method,
                    "alpha": alpha,

                    "TE_sem_original": te_orig,
                    "TE_sem_after": te_after,
                    "TE_sem_delta": te_orig - te_after,
                    "TE_sem_reduction_pct": (100.0 * (te_orig - te_after) / te_orig if te_orig > 1e-6 else np.nan),

                    "BiasGap_original": bg_orig,
                    "BiasGap_after": bg_after,
                    "BiasGap_delta": bg_orig - bg_after if not np.isnan(bg_orig) and not np.isnan(bg_after) else np.nan,
                    "BiasGap_reduction_pct": (
                        100.0 * (bg_orig - bg_after) / max(bg_orig, 1e-8)
                        if not np.isnan(bg_orig) and not np.isnan(bg_after) and bg_orig > 0
                        else np.nan
                    ),

                    "score_d0_after": s0_after,
                    "score_d1_after": s1_after,

                    "quality_drift_mean": drift_mean,
                    "quality_drift_d0": drift0,
                    "quality_drift_d1": drift1,

                    "y_d0_original": y0_orig,
                    "y_d1_original": y1_orig,
                    "y_d0_after": yd0,
                    "y_d1_after": yd1,
                })

                #Save incrementally
                pd.DataFrame(rows).to_csv(mitigation_path, index=False)

    mitigation_df = pd.DataFrame(rows)

display(mitigation_df.head())
print("Mitigation rows:", len(mitigation_df))


In [ ]:
#corpus difference-in-means demographic direction (per head)
#This is corpus-level, so a single pair can't dominate the direction, and both parts get the SAME projection operator 
# (unlike midpoint, which pulls both to a shared point).
 
import numpy as np
import torch
 
def _last_token_head_slice(act: torch.Tensor, h: int) -> torch.Tensor:
    """act: (1, T, hidden). Return (head_dim,) for head h at last position."""
    h0, h1 = h * head_dim, (h + 1) * head_dim
    return act[0, -1, h0:h1].float().cpu()
 
def estimate_demographic_directions(circuit_heads, pairs_subset):
    """
    circuit_heads: list of (layer, head) tuples.
    pairs_subset: DataFrame with prompt_d0 / prompt_d1 columns.
    Returns: dict (l,h) -> unit direction tensor (head_dim,) on model.device,
             plus dict (l,h) -> raw ||d|| norm (for diagnostics).
    """
    layers_needed = sorted(set(l for l, h in circuit_heads))
    # accumulate per-head sums for d0 and d1
    sum0 = {(l, h): torch.zeros(head_dim) for (l, h) in circuit_heads}
    sum1 = {(l, h): torch.zeros(head_dim) for (l, h) in circuit_heads}
    n = 0
    for _, row in pairs_subset.iterrows():
        cache0 = {l: cache_o_proj_input(row["prompt_d0"], l) for l in layers_needed}
        cache1 = {l: cache_o_proj_input(row["prompt_d1"], l) for l in layers_needed}
        for (l, h) in circuit_heads:
            sum0[(l, h)] += _last_token_head_slice(cache0[l], h)
            sum1[(l, h)] += _last_token_head_slice(cache1[l], h)
        n += 1
    dirs, norms = {}, {}
    for (l, h) in circuit_heads:
        d = (sum1[(l, h)] - sum0[(l, h)]) / max(n, 1)
        nrm = float(d.norm())
        norms[(l, h)] = nrm
        d_hat = d / (nrm + 1e-8)
        dirs[(l, h)] = d_hat.to(model.device)
    return dirs, norms
 

In [ ]:
#updated: projection intervention hook + final-token-JSD outcome
 
@torch.no_grad()
def _final_logits_with_projection(prompt: str, circuit_heads, dirs, alpha: float):
    handles = []
    by_layer = {}
    for (l, h) in circuit_heads:
        by_layer.setdefault(l, []).append(h)
 
    def make_hook(layer_idx):
        heads_here = by_layer[layer_idx]
        def pre_hook(mod, inputs):
            x = inputs[0]
            x_new = x.clone()
            for h in heads_here:
                h0, h1 = h * head_dim, (h + 1) * head_dim
                d_hat = dirs[(layer_idx, h)].to(x_new.dtype)  # (head_dim,)
                seg = x_new[:, :, h0:h1]                       # (1,T,head_dim)
                coeff = torch.einsum("btd,d->bt", seg.float(), d_hat.float())  # (1,T)
                seg = seg - alpha * coeff.unsqueeze(-1).to(seg.dtype) * d_hat
                x_new[:, :, h0:h1] = seg
            return (x_new,) + inputs[1:]
        return pre_hook
 
    for l in by_layer:
        handles.append(get_attn_output_module(layers[l]).register_forward_pre_hook(make_hook(l)))
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model(**inputs)
    for hd in handles:
        hd.remove()
    return out.logits[:, -1, :].detach()
 
 
@torch.no_grad()
def _final_logits_with_zero(prompt: str, circuit_heads):
    """Zero-ablation baseline in the SAME final-logits space."""
    handles = []
    by_layer = {}
    for (l, h) in circuit_heads:
        by_layer.setdefault(l, []).append(h)
 
    def make_hook(layer_idx):
        heads_here = by_layer[layer_idx]
        def pre_hook(mod, inputs):
            x = inputs[0]
            x_new = x.clone()
            for h in heads_here:
                h0, h1 = h * head_dim, (h + 1) * head_dim
                x_new[:, :, h0:h1] = 0
            return (x_new,) + inputs[1:]
        return pre_hook
 
    for l in by_layer:
        handles.append(get_attn_output_module(layers[l]).register_forward_pre_hook(make_hook(l)))
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model(**inputs)
    for hd in handles:
        hd.remove()
    return out.logits[:, -1, :].detach()
 
 
@torch.no_grad()
def arm_gap_jsd(row, circuit_heads, dirs, alpha, mode):
    """
    JSD between d0-arm and d1-arm final-token distributions.
    """
    if mode == "none":
        lg0 = get_logits(row["prompt_d0"])
        lg1 = get_logits(row["prompt_d1"])
    elif mode == "projection":
        lg0 = _final_logits_with_projection(row["prompt_d0"], circuit_heads, dirs, alpha)
        lg1 = _final_logits_with_projection(row["prompt_d1"], circuit_heads, dirs, alpha)
    elif mode == "zero":
        lg0 = _final_logits_with_zero(row["prompt_d0"], circuit_heads)
        lg1 = _final_logits_with_zero(row["prompt_d1"], circuit_heads)
    else:
        raise ValueError(mode)
    return js_divergence_from_logits(lg0, lg1)
 

In [ ]:
#random-head null control
 
def sample_random_heads(k, exclude=None, seed=0):
    rng = np.random.default_rng(seed)
    exclude = set(exclude or [])
    all_heads = [(l, h) for l in range(num_layers) for h in range(num_heads)]
    pool = [x for x in all_heads if x not in exclude]
    idx = rng.choice(len(pool), size=min(k, len(pool)), replace=False)
    return [pool[i] for i in idx]
 
def random_unit_directions(circuit_heads, seed=0):
    rng = np.random.default_rng(seed)
    dirs = {}
    for (l, h) in circuit_heads:
        v = torch.tensor(rng.standard_normal(head_dim), dtype=torch.float32)
        v = v / (v.norm() + 1e-8)
        dirs[(l, h)] = v.to(model.device)
    return dirs

In [ ]:
#Shared-circuit intervention: K sweep
shared_path = os.path.join(OUTPUT_DIR, f"shared_circuit_results_{K_TAG}.csv")

def _shared_checkpoint_complete(path):
    if not os.path.exists(path):
        return False
    try:
        d = pd.read_csv(path)
    except Exception:
        return False
    required = {"K", "pair_id", "method", "alpha"}
    if not required.issubset(d.columns):
        return False
    expected = 0
    for _K in TOP_K_LIST:
        if all_top_heads[_K]["shared"]:
            expected += pairs_df["pair_id"].nunique() * 3
    keys = d[["K", "pair_id", "method"]].drop_duplicates()
    return len(keys) == expected

if _shared_checkpoint_complete(shared_path):
    shared_circuit_df = pd.read_csv(shared_path)
else:
    rows = []
    base_lookup = baseline_df.set_index("pair_id")

    for K in TOP_K_LIST:
        shared_heads_k = all_top_heads[K]["shared"]
        jaccard_k = all_top_heads[K]["jaccard_race_sex"]

        print(f"\nShared intervention for K={K}")
        print("shared heads:", shared_heads_k)
        print("jaccard:", jaccard_k)

        if len(shared_heads_k) == 0:
            print(f"No shared heads for K={K}; skipping.")
            continue

        for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc=f"Shared K={K}"):
            pair_id = row["pair_id"]
            heads = shared_heads_k

            cache0 = build_layer_caches(row["prompt_d0"], heads)
            cache1 = build_layer_caches(row["prompt_d1"], heads)
            neutral = build_neutral_caches(cache0, cache1)
            direction = build_directional_caches(cache0, cache1)

            base = base_lookup.loc[pair_id]
            y0_orig, y1_orig = str(base["y_d0"]), str(base["y_d1"])
            te_orig = float(base["TE_sem_original"])
            bg_orig = float(base["BiasGap_original"]) if not pd.isna(base["BiasGap_original"]) else np.nan

            outputs = {
                "shared_zero_ablation": (
                    generate_with_multi_head_intervention(row["prompt_d0"], heads, "zero_ablation"),
                    generate_with_multi_head_intervention(row["prompt_d1"], heads, "zero_ablation"),
                ),
                "shared_midpoint_intervention": (
                    generate_with_multi_head_intervention(row["prompt_d0"], heads, "midpoint_intervention", neutral_caches=neutral),
                    generate_with_multi_head_intervention(row["prompt_d1"], heads, "midpoint_intervention", neutral_caches=neutral),
                ),
                "shared_directional_intervention": (
                    generate_with_multi_head_intervention(row["prompt_d0"], heads, "directional_intervention", direction_caches=direction, alpha=DIRECTION_ALPHA, direction_sign=+1.0),
                    generate_with_multi_head_intervention(row["prompt_d1"], heads, "directional_intervention", direction_caches=direction, alpha=DIRECTION_ALPHA, direction_sign=-1.0),
                ),
            }

            for method, (yd0, yd1) in outputs.items():
                te_after = compute_te_sem(yd0, yd1)
                bg_after, s0_after, s1_after = compute_bias_gap(yd0, yd1)
                drift_mean, drift0, drift1 = compute_quality_drift(y0_orig, y1_orig, yd0, yd1)

                rows.append({
                    "K": K,
                    "pair_id": pair_id,
                    "seed_id": row["seed_id"],
                    "attribute": row["attribute"],
                    "comparison": row["comparison"],
                    "method": method,
                    "alpha":alpha,
                    "num_shared_heads": len(heads),
                    "shared_heads": str(heads),
                    "jaccard_race_sex": jaccard_k,

                    "TE_sem_original": te_orig,
                    "TE_sem_after": te_after,
                    "TE_sem_delta": te_orig - te_after,
                    "TE_sem_reduction_pct": (100.0 * (te_orig - te_after) / te_orig if te_orig > 1e-6 else np.nan),

                    "BiasGap_original": bg_orig,
                    "BiasGap_after": bg_after,
                    "BiasGap_delta": bg_orig - bg_after if not np.isnan(bg_orig) and not np.isnan(bg_after) else np.nan,
                    "BiasGap_reduction_pct": (
                        100.0 * (bg_orig - bg_after) / max(bg_orig, 1e-8)
                        if not np.isnan(bg_orig) and not np.isnan(bg_after) and bg_orig > 0
                        else np.nan
                    ),

                    "quality_drift_mean": drift_mean,
                    "quality_drift_d0": drift0,
                    "quality_drift_d1": drift1,

                    "score_d0_after": s0_after,
                    "score_d1_after": s1_after,
                    "y_d0_after": yd0,
                    "y_d1_after": yd1,
                })

                # Save incrementally in case the job is interrupted.
                pd.DataFrame(rows).to_csv(shared_path, index=False)

    shared_circuit_df = pd.DataFrame(rows)

display(shared_circuit_df.head() if len(shared_circuit_df) else shared_circuit_df)
print("Shared rows:", len(shared_circuit_df))


In [ ]:
#Summary + figures
def summarize(df, group_cols):
    if len(df) == 0:
        return pd.DataFrame()

    out = df.groupby(group_cols, as_index=False).agg(
        n=("pair_id", "nunique"),
        TE_sem_original_mean=("TE_sem_original", "mean"),
        TE_sem_after_mean=("TE_sem_after", "mean"),
        TE_sem_delta_mean=("TE_sem_delta", "mean"),
        BiasGap_original_mean=("BiasGap_original", "mean"),
        BiasGap_after_mean=("BiasGap_after", "mean"),
        BiasGap_delta_mean=("BiasGap_delta", "mean"),
        quality_drift_mean=("quality_drift_mean", "mean"),
    )

    out["TE_sem_reduction_pct_mean"] = (
        100.0 * out["TE_sem_delta_mean"] / out["TE_sem_original_mean"].clip(lower=1e-8)
    )
    out["BiasGap_reduction_pct_mean"] = (
        100.0 * out["BiasGap_delta_mean"] / out["BiasGap_original_mean"].clip(lower=1e-8)
    )
    return out


summary_attr_k = summarize(mitigation_df, ["K", "attribute", "method"])
summary_comp_k = summarize(mitigation_df, ["K", "attribute", "comparison", "method"])
shared_summary_attr_k = (
    summarize(shared_circuit_df, ["K", "attribute", "method"])
    if len(shared_circuit_df)
    else pd.DataFrame()
)

summary_attr_k.to_csv(os.path.join(OUTPUT_DIR, f"summary_by_K_attribute_method_{K_TAG}.csv"), index=False)
summary_comp_k.to_csv(os.path.join(OUTPUT_DIR, f"summary_by_K_comparison_method_{K_TAG}.csv"), index=False)
if len(shared_summary_attr_k):
    shared_summary_attr_k.to_csv(os.path.join(OUTPUT_DIR, f"shared_summary_by_K_attribute_method_{K_TAG}.csv"), index=False)

display(summary_attr_k)
display(shared_summary_attr_k)


# Optional collapsed summaries across K, mainly for quick inspection.
summary_attr = summarize(mitigation_df, ["attribute", "method"])
shared_summary_attr = (
    summarize(shared_circuit_df, ["attribute", "method"])
    if len(shared_circuit_df)
    else pd.DataFrame()
)
summary_attr.to_csv(os.path.join(OUTPUT_DIR, f"summary_collapsed_attribute_method_{K_TAG}.csv"), index=False)
if len(shared_summary_attr):
    shared_summary_attr.to_csv(os.path.join(OUTPUT_DIR, f"shared_summary_collapsed_attribute_method_{K_TAG}.csv"), index=False)


# Heatmaps / tradeoff plots
def plot_nie_heatmap(df, attribute, title, save_path):
    d = df[df["attribute"].str.lower() == attribute].copy()
    if len(d) == 0:
        return
    heat = (
        d.groupby(["layer", "head"])["NIE"]
        .mean()
        .reset_index()
        .pivot(index="layer", columns="head", values="NIE")
        .fillna(0.0)
    )
    plt.figure(figsize=(10, 6))
    plt.imshow(heat.values, aspect="auto")
    plt.colorbar(label="Mean NIE (JSD)")
    plt.xlabel("Head")
    plt.ylabel("Layer")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_tradeoff(summary, title, save_path):
    if len(summary) == 0:
        return

    plt.figure(figsize=(8, 6))
    for _, r in summary.iterrows():
        plt.scatter(r["quality_drift_mean"], r["BiasGap_reduction_pct_mean"])
        label = f"K={int(r['K'])} {r['attribute']} / {r['method']}" if "K" in summary.columns else f"{r['attribute']} / {r['method']}"
        plt.annotate(
            label,
            (r["quality_drift_mean"], r["BiasGap_reduction_pct_mean"]),
            fontsize=7,
            xytext=(4, 4),
            textcoords="offset points",
        )
    plt.axhline(0, linewidth=1)
    plt.xlabel("Quality drift (lower is better)")
    plt.ylabel("BiasGap reduction (%)")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_k_sweep(summary, metric_col, ylabel, title, save_path):
    if len(summary) == 0:
        return

    plt.figure(figsize=(8, 5))
    for (attribute, method), sub in summary.groupby(["attribute", "method"]):
        sub = sub.sort_values("K")
        plt.plot(sub["K"], sub[metric_col], marker="o", label=f"{attribute}/{method}")
    plt.axhline(0, linewidth=1)
    plt.xlabel("Top-K mediator heads")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(fontsize=7, ncol=1)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


plot_nie_heatmap(nie_df, "race", "Race NIE heatmap", os.path.join(OUTPUT_DIR, "fig_race_NIE_heatmap.png"))
plot_nie_heatmap(nie_df, "sex", "Sex NIE heatmap", os.path.join(OUTPUT_DIR, "fig_sex_NIE_heatmap.png"))

plot_tradeoff(
    summary_attr_k,
    "Mitigation tradeoff: BiasGap reduction vs quality drift",
    os.path.join(OUTPUT_DIR, f"fig_biasgap_vs_quality_drift_{K_TAG}.png"),
)

plot_k_sweep(
    summary_attr_k,
    "BiasGap_reduction_pct_mean",
    "BiasGap reduction (%)",
    "BiasGap reduction across Top-K mediator sizes",
    os.path.join(OUTPUT_DIR, f"fig_k_sweep_biasgap_reduction_{K_TAG}.png"),
)

plot_k_sweep(
    summary_attr_k,
    "TE_sem_reduction_pct_mean",
    "TE_sem reduction (%)",
    "Semantic TE reduction across Top-K mediator sizes",
    os.path.join(OUTPUT_DIR, f"fig_k_sweep_te_sem_reduction_{K_TAG}.png"),
)

if len(shared_summary_attr_k):
    plot_k_sweep(
        shared_summary_attr_k,
        "BiasGap_reduction_pct_mean",
        "BiasGap reduction (%)",
        "Shared-circuit BiasGap reduction across Top-K mediator sizes",
        os.path.join(OUTPUT_DIR, f"fig_shared_k_sweep_biasgap_reduction_{K_TAG}.png"),
    )


In [ ]:
# directions- one-forward multi-layer caching, nested-K reuse,
import os, json
import numpy as np
import torch
 
ALPHA_LIST = [0.5, 1.0]         
DIRECTION_SPLIT_HALF = False     
                                
 
def cache_o_proj_inputs_multi(prompt: str, layer_list):
    """Cache o_proj inputs for ALL requested layers in a single forward pass.
    Returns dict layer_idx -> (1, T, hidden) tensor (detached, on device)."""
    caches, handles = {}, []
    def make_hook(l):
        def pre_hook(mod, inputs):
            caches[l] = inputs[0].detach()
        return pre_hook
    for l in sorted(set(layer_list)):
        handles.append(get_attn_output_module(layers[l]).register_forward_pre_hook(make_hook(l)))
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        _ = model(**inputs)
    for h in handles:
        h.remove()
    return caches
 
 
def estimate_demographic_directions_fast(circuit_heads, pairs_subset):
    """Corpus difference-in-means direction per head, last prompt-token position.
    One forward per part per pair (all layers hooked at once)."""
    layers_needed = sorted(set(l for l, h in circuit_heads))
    sum0 = {(l, h): torch.zeros(head_dim) for (l, h) in circuit_heads}
    sum1 = {(l, h): torch.zeros(head_dim) for (l, h) in circuit_heads}
    n = 0
    for _, row in pairs_subset.iterrows():
        c0 = cache_o_proj_inputs_multi(row["prompt_d0"], layers_needed)
        c1 = cache_o_proj_inputs_multi(row["prompt_d1"], layers_needed)
        for (l, h) in circuit_heads:
            h0, h1 = h * head_dim, (h + 1) * head_dim
            sum0[(l, h)] += c0[l][0, -1, h0:h1].float().cpu()
            sum1[(l, h)] += c1[l][0, -1, h0:h1].float().cpu()
        n += 1
    dirs, norms = {}, {}
    for (l, h) in circuit_heads:
        d = (sum1[(l, h)] - sum0[(l, h)]) / max(n, 1)
        nrm = float(d.norm())
        norms[(l, h)] = nrm
        dirs[(l, h)] = (d / (nrm + 1e-8)).to(model.device)
    return dirs, norms
 
 
def split_pairs_for_direction(sub):
    seeds = sorted(sub["seed_id"].unique())
    rng = np.random.default_rng(SEED)
    rng.shuffle(seeds)
    est_seeds = set(seeds[: len(seeds) // 2])
    est = sub[sub["seed_id"].isin(est_seeds)].reset_index(drop=True)
    ev  = sub[~sub["seed_id"].isin(est_seeds)].reset_index(drop=True)
    return est, ev
 
 
#estimate once per attribute at k_max; smaller K just slice (nested top-K)
K_MAX = max(TOP_K_LIST)
DIR_TAG = "splithalf" if DIRECTION_SPLIT_HALF else "full"
 
directions_by_attr = {}    # attr -> {"dirs": {(l,h): d_hat}, "norms": {...},
                           #          "est_df": ..., "eval_df": ...}
for attribute in ["race", "sex"]:
    sub = pairs_df[pairs_df["attribute"].str.lower() == attribute].reset_index(drop=True)
    if len(sub) == 0:
        continue
    circuit_max = choose_circuit_for_k(attribute, K_MAX)
    if DIRECTION_SPLIT_HALF:
        est_df, eval_df = split_pairs_for_direction(sub)
    else:
        est_df, eval_df = sub, sub
    print(f"[{attribute}] estimating d_hat on {len(est_df)} pairs "
          f"for {len(circuit_max)} heads (K_MAX={K_MAX}, {DIR_TAG})")
    dirs, norms = estimate_demographic_directions_fast(circuit_max, est_df)
    directions_by_attr[attribute] = {
        "dirs": dirs, "norms": norms, "est_df": est_df, "eval_df": eval_df,
    }
    # persist for reproducibility / appendix
    npz_path = os.path.join(OUTPUT_DIR, f"directions_{attribute}_Kmax{K_MAX}_{DIR_TAG}.npz")
    np.savez(
        npz_path,
        heads=np.array([f"{l}.{h}" for (l, h) in dirs]),
        dirs=np.stack([dirs[k].cpu().numpy() for k in dirs]),
        norms=np.array([norms[k] for k in dirs]),
        n_est_pairs=len(est_df),
    )
    print("  saved:", npz_path)

In [ ]:
# updated-JSD outcome for all four modes
@torch.no_grad()
def _final_logits_with_projection(prompt, circuit_heads, dirs, alpha):
    handles, by_layer = [], {}
    for (l, h) in circuit_heads:
        by_layer.setdefault(l, []).append(h)
    def make_hook(layer_idx):
        heads_here = by_layer[layer_idx]
        def pre_hook(mod, inputs):
            x = inputs[0]
            x_new = x.clone()
            for h in heads_here:
                h0, h1 = h * head_dim, (h + 1) * head_dim
                d_hat = dirs[(layer_idx, h)].to(x_new.dtype)
                seg = x_new[:, :, h0:h1]
                coeff = torch.einsum("btd,d->bt", seg.float(), d_hat.float())
                x_new[:, :, h0:h1] = seg - alpha * coeff.unsqueeze(-1).to(seg.dtype) * d_hat
            return (x_new,) + inputs[1:]
        return pre_hook
    for l in by_layer:
        handles.append(get_attn_output_module(layers[l]).register_forward_pre_hook(make_hook(l)))
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model(**inputs)
    for hd in handles:
        hd.remove()
    return out.logits[:, -1, :].detach()
 
 
@torch.no_grad()
def _final_logits_with_zero(prompt, circuit_heads):
    handles, by_layer = [], {}
    for (l, h) in circuit_heads:
        by_layer.setdefault(l, []).append(h)
    def make_hook(layer_idx):
        heads_here = by_layer[layer_idx]
        def pre_hook(mod, inputs):
            x = inputs[0]
            x_new = x.clone()
            for h in heads_here:
                h0, h1 = h * head_dim, (h + 1) * head_dim
                x_new[:, :, h0:h1] = 0
            return (x_new,) + inputs[1:]
        return pre_hook
    for l in by_layer:
        handles.append(get_attn_output_module(layers[l]).register_forward_pre_hook(make_hook(l)))
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model(**inputs)
    for hd in handles:
        hd.remove()
    return out.logits[:, -1, :].detach()
 
 
@torch.no_grad()
def _final_logits_with_cache_patch(prompt, circuit_heads, patch_caches):
    """Patch each circuit head's slice with the given per-layer cache"""
    handles, by_layer = [], {}
    for (l, h) in circuit_heads:
        by_layer.setdefault(l, []).append(h)
    def make_hook(layer_idx):
        heads_here = by_layer[layer_idx]
        def pre_hook(mod, inputs):
            x = inputs[0]
            x_new = x.clone()
            src = patch_caches[layer_idx]
            T = min(x_new.shape[1], src.shape[1])
            for h in heads_here:
                h0, h1 = h * head_dim, (h + 1) * head_dim
                x_new[:, :T, h0:h1] = src[:, :T, h0:h1].to(x_new.device, dtype=x_new.dtype)
            return (x_new,) + inputs[1:]
        return pre_hook
    for l in by_layer:
        handles.append(get_attn_output_module(layers[l]).register_forward_pre_hook(make_hook(l)))
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model(**inputs)
    for hd in handles:
        hd.remove()
    return out.logits[:, -1, :].detach()
 
 
def pair_jsd_all_modes(row, circuit_heads, dirs, alpha_list,
                       null_heads=None, null_dirs=None,
                       midpoint_caches=None, base_logits=None):
    """
    For one pair and one circuit (a given K), compute the counterpart gap JSD and the
    within-counterfactual drift for every mode. Reuses baseline logits across everything.
 
    Returns list of dicts:{mode, alpha, gap_after, drift_d0, drift_d1}
    Modes: zero, midpoint (alpha=NaN), projection, null_projection (per alpha).
    """
    if base_logits is None:
        lg0_b = get_logits(row["prompt_d0"])
        lg1_b = get_logits(row["prompt_d1"])
    else:
        lg0_b, lg1_b = base_logits
 
    out = []
 
    def emit(mode, alpha, lg0, lg1):
        out.append({
            "mode": mode,
            "alpha": alpha,
            "gap_after": js_divergence_from_logits(lg0, lg1),
            "drift_d0": js_divergence_from_logits(lg0_b, lg0),
            "drift_d1": js_divergence_from_logits(lg1_b, lg1),
        })
 
    # zero-ablation (alpha-free)
    lg0 = _final_logits_with_zero(row["prompt_d0"], circuit_heads)
    lg1 = _final_logits_with_zero(row["prompt_d1"], circuit_heads)
    emit("zero", np.nan, lg0, lg1)
 
    # midpoint strawman (alpha-free)
    if midpoint_caches is not None:
        lg0 = _final_logits_with_cache_patch(row["prompt_d0"], circuit_heads, midpoint_caches)
        lg1 = _final_logits_with_cache_patch(row["prompt_d1"], circuit_heads, midpoint_caches)
        emit("midpoint", np.nan, lg0, lg1)
 
    # projection + matched random null, per alpha
    for alpha in alpha_list:
        lg0 = _final_logits_with_projection(row["prompt_d0"], circuit_heads, dirs, alpha)
        lg1 = _final_logits_with_projection(row["prompt_d1"], circuit_heads, dirs, alpha)
        emit("projection", alpha, lg0, lg1)
        if null_heads is not None:
            lg0 = _final_logits_with_projection(row["prompt_d0"], null_heads, null_dirs, alpha)
            lg1 = _final_logits_with_projection(row["prompt_d1"], null_heads, null_dirs, alpha)
            emit("null_projection", alpha, lg0, lg1)
 
    return out, (lg0_b, lg1_b)
 

In [ ]:
# MAIN EXPERIMENT: K - alpha sweep, per-pair rows

import pandas as pd
 
sweep_path = os.path.join(
    OUTPUT_DIR, f"projection_jsd_sweep_{K_TAG}_{DIR_TAG}.csv"
)
 
def _atomic_save(df, path):
    tmp = path + ".tmp"
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)         
 
if os.path.exists(sweep_path):
    done_df = pd.read_csv(sweep_path)
    rows = done_df.to_dict("records")
    done_pairs = set(zip(done_df["attribute"], done_df["pair_id"]))
    print(f"Resuming: {len(done_df)} rows, {len(done_pairs)} (attr, pair) done.")
else:
    rows, done_pairs = [], set()
 
SAVE_EVERY = 5   # checkpoint every N pairs (atomic full rewrite; cheap at this size)
 
for attribute in ["race", "sex"]:
    if attribute not in directions_by_attr:
        continue
    info = directions_by_attr[attribute]
    eval_df = info["eval_df"]
    dirs_max = info["dirs"]
 
    # circuits per K (nested), matched random nulls per K (fixed seed)
    circuits, nulls = {}, {}
    for K in TOP_K_LIST:
        c = choose_circuit_for_k(attribute, K)
        circuits[K] = c
        nh = sample_random_heads(len(c), exclude=set(c), seed=SEED + K)
        nulls[K] = (nh, random_unit_directions(nh, seed=SEED + K))
 
    layers_max = sorted(set(l for l, h in circuits[max(TOP_K_LIST)]))
 
    since_save = 0
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df),
                       desc=f"JSD sweep [{attribute}]"):
        if (attribute, row["pair_id"]) in done_pairs:
            continue
 
        # baseline logits: once per pair, reused for every K/alpha/mode
        base_logits = (get_logits(row["prompt_d0"]), get_logits(row["prompt_d1"]))
        gap_base = js_divergence_from_logits(*base_logits)
 
        # midpoint caches: one forward per arm at K_MAX layers, sliced per K
        c0 = cache_o_proj_inputs_multi(row["prompt_d0"], layers_max)
        c1 = cache_o_proj_inputs_multi(row["prompt_d1"], layers_max)
        mid_caches = {}
        for l in layers_max:
            T = min(c0[l].shape[1], c1[l].shape[1])
            m = c0[l].clone()
            m[:, -T:, :] = 0.5 * (c0[l][:, -T:, :] + c1[l][:, -T:, :])
            mid_caches[l] = m
 
        for K in TOP_K_LIST:
            circuit = circuits[K]
            if len(circuit) == 0:
                continue
            dirs_k = {hk: dirs_max[hk] for hk in circuit}   # nested slice
            null_heads, null_dirs = nulls[K]
 
            mode_rows, _ = pair_jsd_all_modes(
                row, circuit, dirs_k, ALPHA_LIST,
                null_heads=null_heads, null_dirs=null_dirs,
                midpoint_caches=mid_caches, base_logits=base_logits,
            )
            for m in mode_rows:
                rows.append({
                    "model_tag": MODEL_TAG,
                    "attribute": attribute,
                    "K": K,
                    "alpha": m["alpha"],
                    "mode": m["mode"],
                    "pair_id": row["pair_id"],
                    "seed_id": row["seed_id"],
                    "comparison": row["comparison"],
                    "group0": row["group0"],
                    "group1": row["group1"],
                    "n_heads": len(circuit),
                    "gap_base": gap_base,
                    "gap_after": m["gap_after"],
                    "reduction": gap_base - m["gap_after"],
                    "drift_d0": m["drift_d0"],
                    "drift_d1": m["drift_d1"],
                    "drift_mean": 0.5 * (m["drift_d0"] + m["drift_d1"]),
                    "direction_split": DIR_TAG,
                })
 
        done_pairs.add((attribute, row["pair_id"]))
        since_save += 1
        if since_save >= SAVE_EVERY:
            _atomic_save(pd.DataFrame(rows), sweep_path)
            since_save = 0
 
    _atomic_save(pd.DataFrame(rows), sweep_path)
 
sweep_df = pd.DataFrame(rows)
_atomic_save(sweep_df, sweep_path)
print("Sweep rows:", len(sweep_df), "->", sweep_path)
 

In [ ]:
# full-generation quality drift at the operating point 
@torch.no_grad()
def generate_with_projection(prompt, circuit_heads, dirs, alpha,
                             max_new_tokens=MAX_NEW_TOKENS):
    handles, by_layer = [], {}
    for (l, h) in circuit_heads:
        by_layer.setdefault(l, []).append(h)
    def make_hook(layer_idx):
        heads_here = by_layer[layer_idx]
        def pre_hook(mod, inputs):
            x = inputs[0]
            x_new = x.clone()
            for h in heads_here:
                h0, h1 = h * head_dim, (h + 1) * head_dim
                d_hat = dirs[(layer_idx, h)].to(x_new.dtype)
                seg = x_new[:, :, h0:h1]
                coeff = torch.einsum("btd,d->bt", seg.float(), d_hat.float())
                x_new[:, :, h0:h1] = seg - alpha * coeff.unsqueeze(-1).to(seg.dtype) * d_hat
            return (x_new,) + inputs[1:]
        return pre_hook
    for l in by_layer:
        handles.append(get_attn_output_module(layers[l]).register_forward_pre_hook(make_hook(l)))
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    gen_kwargs = dict(max_new_tokens=max_new_tokens, do_sample=DO_SAMPLE,
                      pad_token_id=tokenizer.eos_token_id)
    out = model.generate(**inputs, **gen_kwargs)
    for hd in handles:
        hd.remove()
    new_ids = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()
 
 
op_rows = []
op_path = os.path.join(OUTPUT_DIR, f"projection_generation_at_op_{DIR_TAG}.csv")
base_lookup = baseline_df.set_index("pair_id")
 
for _, r in op.iterrows():
    attribute, K_star, a_star = r["attribute"], int(r["K"]), float(r["alpha"])
    info = directions_by_attr[attribute]
    circuit = choose_circuit_for_k(attribute, K_star)
    dirs_k = {hk: info["dirs"][hk] for hk in circuit}
    sub = info["eval_df"]
    for _, row in tqdm(sub.iterrows(), total=len(sub),
                       desc=f"gen@op [{attribute}] K={K_star} a={a_star}"):
        yd0 = generate_with_projection(row["prompt_d0"], circuit, dirs_k, a_star)
        yd1 = generate_with_projection(row["prompt_d1"], circuit, dirs_k, a_star)
        base = base_lookup.loc[row["pair_id"]]
        drift_mean, d0, d1 = compute_quality_drift(
            str(base["y_d0"]), str(base["y_d1"]), yd0, yd1)
        te_after = compute_te_sem(yd0, yd1)
        op_rows.append({
            "attribute": attribute, "K": K_star, "alpha": a_star,
            "pair_id": row["pair_id"],
            "TE_sem_original": float(base["TE_sem_original"]),
            "TE_sem_after_projection": te_after,
            "quality_drift_mean": drift_mean,
            "y_d0_after": yd0, "y_d1_after": yd1,
        })
        if len(op_rows) % 5 == 0:
            _atomic_save(pd.DataFrame(op_rows), op_path)
 
_atomic_save(pd.DataFrame(op_rows), op_path)
print("Saved:", op_path)